In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aegis_fraud_workspace.finguard;

CREATE VOLUME IF NOT EXISTS aegis_fraud_workspace.finguard.finguard_volume;

In [0]:
import os

BASE_PATH = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume"

PATHS = {
    "landing": f"{BASE_PATH}/raw_landing",
    "checkpoints": f"{BASE_PATH}/_checkpoints",
    "models": f"{BASE_PATH}/artifacts/models",
    "schema": f"{BASE_PATH}/_checkpoints/schema_registry"
}

for name, path in PATHS.items():
    dbutils.fs.mkdirs(path)

print("[INFO] Working directories created successfully.")

[INFO] Working directories created successfully.


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

customer_seed = [
    ("USR_001", "Aarav Sharma", "Hyderabad", 5000.0, "TIER_1"),
    ("USR_002", "Priya Nair", "Bengaluru", 2500.0, "TIER_2"),
    ("USR_003", "Rohan Mehta", "Mumbai", 7500.0, "TIER_1"),
    ("USR_004", "Sneha Rao", "Hyderabad", 1500.0, "TIER_3"),
    ("USR_005", "Vikram Verma", "London", 10000.0, "PRIVATE_WEALTH"),
    ("USR_006", "Ananya Das", "Bengaluru", 3000.0, "TIER_2"),
    ("USR_007", "Karthik Reddy", "Hyderabad", 6000.0, "TIER_1"),
    ("USR_008", "Neha Kulkarni", "Pune", 2000.0, "TIER_3"),
    ("USR_009", "Arjun Patel", "Mumbai", 4500.0, "TIER_2"),
    ("USR_010", "Divya Iyer", "Chennai", 3500.0, "TIER_2"),
]

dim_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("customer_name", StringType(), False),
    StructField("home_city", StringType(), False),
    StructField("daily_limit", DoubleType(), False),
    StructField("segment", StringType(), True)
])

df_dim_customers = spark.createDataFrame(customer_seed, schema=dim_schema)

(
    df_dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_customers")
)

print(f"[SUCCESS] dim_customers table loaded. Row count: {spark.table('dim_customers').count()}")

[SUCCESS] dim_customers table loaded. Row count: 10
